<a href="https://colab.research.google.com/github/dineshrayella/NLP/blob/main/ASS_13_4_2403a54099.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Perform text classification on the dataset found in the zip file `/content/archive (10).zip` using a Multinomial Naive Bayes classifier, including data extraction, preprocessing, feature engineering, model training, evaluation, and result analysis.

## Extract Data

### Subtask:
Extract the contents of the provided zip file `/content/archive (10).zip` to access the text classification dataset.


**Reasoning**:
To extract the contents of the zip file, I will use the `zipfile` module. This involves importing the module, defining the path to the zip file and the extraction directory, and then using `ZipFile` to extract all contents.



In [1]:
import zipfile
import os

# Define the path to the zip file
zip_file_path = '/content/archive (10).zip'

# Define the extraction directory
extraction_dir = '/content/extracted_data'

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_dir, exist_ok=True)

# Open the zip file and extract its contents
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_dir)

print(f"Contents of '{zip_file_path}' extracted to '{extraction_dir}'")

# List the contents of the extracted directory to verify
print("Extracted files and directories:")
for root, dirs, files in os.walk(extraction_dir):
    for name in files:
        print(os.path.join(root, name))
    for name in dirs:
        print(os.path.join(root, name))

Contents of '/content/archive (10).zip' extracted to '/content/extracted_data'
Extracted files and directories:
/content/extracted_data/spam.csv


## Load Dataset and Sample

### Subtask:
Load the text classification dataset from the extracted files into a pandas DataFrame. Display sample text-label pairs to understand the data structure.


**Reasoning**:
I need to load the `spam.csv` file into a pandas DataFrame. I will import pandas, construct the full file path using the `extraction_dir` variable, read the CSV, and then display the first few rows and column names of the DataFrame.



In [2]:
import pandas as pd

# Construct the full path to the spam.csv file
csv_file_path = os.path.join(extraction_dir, 'spam.csv')

# Load the CSV file into a pandas DataFrame, specifying encoding if necessary
try:
    df = pd.read_csv(csv_file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(csv_file_path, encoding='latin-1')

# Display the first few rows of the DataFrame
print("First 5 rows of the DataFrame:")
print(df.head())

# Print the column names
print("\nColumn names:")
print(df.columns)

First 5 rows of the DataFrame:
     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  

Column names:
Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')


## Text Preprocessing

### Subtask:
Perform text preprocessing steps: convert text to lowercase, remove punctuation and special characters, and optionally remove stopwords to clean the text data.


**Reasoning**:
The first instruction is to rename columns and drop unnecessary ones. This will prepare the DataFrame for text preprocessing by giving meaningful names to the relevant columns and removing irrelevant ones. After that, I will convert the 'text' column to lowercase as a standard text preprocessing step.



In [3]:
import string
import re

# 1. Rename columns 'v1' to 'label' and 'v2' to 'text'
df = df.rename(columns={'v1': 'label', 'v2': 'text'})

# 2. Drop the columns 'Unnamed: 2', 'Unnamed: 3', and 'Unnamed: 4'
df = df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], errors='ignore')

# 3. Convert all text in the 'text' column to lowercase
df['text'] = df['text'].str.lower()

# 4. Remove punctuation from the 'text' column
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['text'] = df['text'].apply(remove_punctuation)

# 5. Remove digits and special characters (keeping only letters and spaces)
def remove_digits_special_chars(text):
    # Keep only letters and spaces
    return re.sub(r'[^a-z\s]', '', text)

df['text'] = df['text'].apply(remove_digits_special_chars)

# 6. Display the first few rows of the preprocessed DataFrame to verify the changes
print("First 5 rows of the preprocessed DataFrame:")
print(df.head())

print("\nColumn names after preprocessing:")
print(df.columns)

First 5 rows of the preprocessed DataFrame:
  label                                               text
0   ham  go until jurong point crazy available only in ...
1   ham                            ok lar joking wif u oni
2  spam  free entry in  a wkly comp to win fa cup final...
3   ham        u dun say so early hor u c already then say
4   ham  nah i dont think he goes to usf he lives aroun...

Column names after preprocessing:
Index(['label', 'text'], dtype='object')


### Remove Stopwords

**Reasoning**:
Stopwords are common words (like 'the', 'is', 'a') that often carry little meaning in text analysis and can be removed to reduce noise and improve efficiency in natural language processing tasks. I will use the `nltk` library to remove English stopwords from the 'text' column of the DataFrame. First, I need to ensure the `stopwords` corpus is downloaded. Then, I'll create a function to filter out stopwords and apply it to the 'text' column.

**Reasoning**:
To remove stopwords, I first need to ensure the `nltk` library is installed and the 'stopwords' corpus is downloaded. Then, I will import `nltk` and `stopwords`, and apply a function to remove these words from the 'text' column of the DataFrame. Finally, I will display the first few rows to show the effect of stopword removal.



In [4]:
import nltk
from nltk.corpus import stopwords

# Download the 'stopwords' corpus if not already downloaded
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')

# Get English stopwords
stop_words = set(stopwords.words('english'))

# Function to remove stopwords
def remove_stopwords(text):
    return ' '.join([word for word in str(text).split() if word not in stop_words])

# Apply the function to the 'text' column
df['text'] = df['text'].apply(remove_stopwords)

# Display the first few rows of the DataFrame after stopword removal
print("First 5 rows of the DataFrame after stopword removal:")
print(df.head())

First 5 rows of the DataFrame after stopword removal:
  label                                               text
0   ham  go jurong point crazy available bugis n great ...
1   ham                            ok lar joking wif u oni
2  spam  free entry wkly comp win fa cup final tkts st ...
3   ham                u dun say early hor u c already say
4   ham        nah dont think goes usf lives around though


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


**Reasoning**:
The text preprocessing steps have been completed. The next logical step, as per the overall task description, is 'Feature Engineering'. This involves transforming the cleaned text data into numerical features that can be used by a machine learning model. I will begin by defining this next subtask.



In [5]:
## Feature Engineering

### Subtask:
Convert the preprocessed text data into numerical feature vectors using techniques like TF-IDF or CountVectorizer, which are essential for machine learning models.

SyntaxError: invalid syntax (1477040683.py, line 4)

**Reasoning**:
The previous code block resulted in a `SyntaxError` because markdown content was incorrectly placed in a python code cell. To define the next subtask, I need to use a markdown cell instead of a python code cell.



In [6]:
## Feature Engineering

### Subtask:
Convert the preprocessed text data into numerical feature vectors using techniques like TF-IDF or CountVectorizer, which are essential for machine learning models.

SyntaxError: invalid syntax (1477040683.py, line 4)

**Reasoning**:
The previous attempt to create a subtask header failed because markdown content was mistakenly placed in a Python cell, causing a `SyntaxError`. To correctly introduce the 'Feature Engineering' subtask, I must ensure the content is within a markdown cell.



In [7]:
## Feature Engineering

### Subtask:
Convert the preprocessed text data into numerical feature vectors using techniques like TF-IDF or CountVectorizer, which are essential for machine learning models.

SyntaxError: invalid syntax (1477040683.py, line 4)

**Reasoning**:
The previous attempts failed because the markdown content was incorrectly placed in a Python code block. To correctly introduce the 'Feature Engineering' subtask, I need to create a markdown cell with the specified content.



In [8]:
## Feature Engineering

### Subtask:
Convert the preprocessed text data into numerical feature vectors using techniques like TF-IDF or CountVectorizer, which are essential for machine learning models.

SyntaxError: invalid syntax (1477040683.py, line 4)

# Task
Convert the preprocessed text data in the 'text' column of the DataFrame `df` into numerical feature vectors using `TfidfVectorizer`, creating a TF-IDF matrix. The 'label' column will be used as the target variable.

## Feature Extraction

### Subtask:
Convert the preprocessed text data in the 'text' column of the DataFrame `df` into numerical feature vectors using `TfidfVectorizer`, creating a TF-IDF matrix. The 'label' column will be used as the target variable.


**Reasoning**:
The subtask requires converting text data into numerical features using `TfidfVectorizer`. I will import the necessary class, instantiate it, and apply it to the 'text' column of the DataFrame. I will also define the target variable 'y' from the 'label' column. Finally, I will print the shape of the feature matrix and the target variable to verify the feature extraction.



In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Instantiate TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting features to avoid very high dimensionality

# Apply fit_transform to the 'text' column to create the TF-IDF matrix
X = tfidf_vectorizer.fit_transform(df['text'])

# Get the target variable (labels)
y = df['label']

print("Shape of the TF-IDF feature matrix (X):", X.shape)
print("Shape of the target variable (y):", y.shape)
print("\nFirst 5 entries of target variable (y):")
print(y.head())

Shape of the TF-IDF feature matrix (X): (5572, 5000)
Shape of the target variable (y): (5572,)

First 5 entries of target variable (y):
0     ham
1     ham
2    spam
3     ham
4     ham
Name: label, dtype: object


## Train-Test Split

### Subtask:
Divide the feature vectors (TF-IDF matrix) and corresponding labels into training and testing sets to prepare for model training and evaluation. This ensures that the model is evaluated on unseen data.


**Reasoning**:
To perform the train-test split as instructed, I need to import `train_test_split` from `sklearn.model_selection`, then apply it to `X` and `y` with the specified `test_size` and `random_state`, and finally print the shapes of the resulting datasets to verify the split.



In [10]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Print the shapes of the resulting sets to verify the split
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (4457, 5000)
Shape of X_test: (1115, 5000)
Shape of y_train: (4457,)
Shape of y_test: (1115,)


## Model Implementation and Training

### Subtask:
Train a Multinomial Naive Bayes classification model using the training feature vectors and training labels. This model is well-suited for text classification tasks with discrete features (like word counts or TF-IDF scores).


**Reasoning**:
To train the Multinomial Naive Bayes model, I will import `MultinomialNB`, instantiate it, and then fit the model using the `X_train` (TF-IDF feature vectors) and `y_train` (labels).



In [11]:
from sklearn.naive_bayes import MultinomialNB

# Instantiate the Multinomial Naive Bayes model
mnb_model = MultinomialNB()

# Train the model using the training data
mnb_model.fit(X_train, y_train)

print("Multinomial Naive Bayes model trained successfully.")

Multinomial Naive Bayes model trained successfully.


## Model Evaluation

### Subtask:
Evaluate the trained Multinomial Naive Bayes model's performance on the test set. This will involve making predictions on the test data and calculating common classification metrics such as accuracy, precision, recall, and F1-score.


**Reasoning**:
To evaluate the model, I need to first make predictions on the test set and then calculate various classification metrics. I will use the trained `mnb_model` to predict `y_pred` from `X_test` and then import `accuracy_score`, `precision_score`, `recall_score`, and `f1_score` from `sklearn.metrics` to compute and print the evaluation metrics.



In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Make predictions on the test set
y_pred = mnb_model.predict(X_test)

# 2. Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label='spam') # Specify pos_label for binary classification
recall = recall_score(y_test, y_pred, pos_label='spam')
f1 = f1_score(y_test, y_pred, pos_label='spam')

# 3. Print the metrics
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Accuracy: 0.9731
Precision: 1.0000
Recall: 0.8000
F1-Score: 0.8889


## Result Analysis

### Subtask:
Interpret the evaluation metrics to understand the model's strengths and weaknesses. This may include examining the confusion matrix and discussing the implications of the model's performance for the text classification task.


**Reasoning**:
To interpret the model's performance, I will import `classification_report` and `confusion_matrix` from `sklearn.metrics`, then generate and print both the classification report and the confusion matrix using the predicted and true labels.



In [13]:
from sklearn.metrics import classification_report, confusion_matrix

# Generate and print the classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Generate and print the confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       965
        spam       1.00      0.80      0.89       150

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115


Confusion Matrix:
[[965   0]
 [ 30 120]]


### Interpretation of Evaluation Metrics

**Classification Report Analysis:**

*   **Accuracy (0.97):** The model correctly classifies approximately 97.3% of the messages. This indicates a generally strong performance.
*   **'Ham' Class:**
    *   **Precision (0.97):** When the model predicts a message is 'ham', it is correct 97% of the time. This is good, meaning few actual spam messages are misclassified as ham (false negatives).
    *   **Recall (1.00):** The model correctly identifies all actual 'ham' messages (100%). This is excellent, indicating no legitimate 'ham' messages are misclassified as spam (false positives).
    *   **F1-Score (0.98):** This is a high score, showing a good balance between precision and recall for the 'ham' class.
*   **'Spam' Class:**
    *   **Precision (1.00):** When the model predicts a message is 'spam', it is correct 100% of the time. This is outstanding, meaning there are no false positives (no 'ham' messages are incorrectly identified as spam).
    *   **Recall (0.80):** The model correctly identifies 80% of all actual 'spam' messages. This means 20% of actual 'spam' messages are missed and classified as 'ham' (false negatives).
    *   **F1-Score (0.89):** This score is good but lower than 'ham', primarily due to the lower recall for the 'spam' class.

**Confusion Matrix Analysis:**

```
[[965   0]
 [ 30 120]]
```
*   **True Negatives (TN): 965** - The model correctly predicted 965 'ham' messages.
*   **False Positives (FP): 0** - The model incorrectly predicted 0 'ham' messages as 'spam'. This aligns with the 100% precision for 'spam'.
*   **False Negatives (FN): 30** - The model incorrectly predicted 30 'spam' messages as 'ham'. These are the missed spam messages.
*   **True Positives (TP): 120** - The model correctly predicted 120 'spam' messages.

**Model's Strengths:**

*   **Excellent at identifying 'ham' messages:** The model has 100% recall for 'ham', meaning it will not flag legitimate messages as spam. This is crucial for user satisfaction.
*   **Perfect precision for 'spam':** When the model says something is spam, it is always correct (100% precision). This means users won't get annoyed by important 'ham' messages being incorrectly filtered as spam.

**Model's Weaknesses:**

*   **Moderate recall for 'spam':** The main weakness is that it misses 20% of actual spam messages (30 false negatives). While 80% recall is decent, it means some spam will still get through.

**Implications for Text Classification (Spam Detection):**

The Multinomial Naive Bayes model performs very well overall for this dataset. Its high precision for spam and perfect recall for ham are highly desirable in a spam filter. Users will not have legitimate emails incorrectly classified as spam, which is often a major pain point. However, the trade-off is that some spam messages will still slip through. Depending on the application, an 80% recall for spam might be acceptable, or it might require further tuning (e.g., adjusting the classification threshold, trying different models, or adding more features) to capture more spam, potentially at the cost of slight precision reduction for spam or recall for ham. For a first pass, this is a strong and user-friendly spam filter due to its low false positive rate.

### Interpretation of Evaluation Metrics

**Classification Report Analysis:**

*   **Accuracy (0.97):** The model correctly classifies approximately 97.3% of the messages. This indicates a generally strong performance.
*   **'Ham' Class:**
    *   **Precision (0.97):** When the model predicts a message is 'ham', it is correct 97% of the time. This is good, meaning few actual spam messages are misclassified as ham (false negatives).
    *   **Recall (1.00):** The model correctly identifies all actual 'ham' messages (100%). This is excellent, indicating no legitimate 'ham' messages are misclassified as spam (false positives).
    *   **F1-Score (0.98):** This is a high score, showing a good balance between precision and recall for the 'ham' class.
*   **'Spam' Class:**
    *   **Precision (1.00):** When the model predicts a message is 'spam', it is correct 100% of the time. This is outstanding, meaning there are no false positives (no 'ham' messages are incorrectly identified as spam).
    *   **Recall (0.80):** The model correctly identifies 80% of all actual 'spam' messages. This means 20% of actual 'spam' messages are missed and classified as 'ham' (false negatives).
    *   **F1-Score (0.89):** This score is good but lower than 'ham', primarily due to the lower recall for the 'spam' class.

**Confusion Matrix Analysis:**

```
[[965   0]
 [ 30 120]]
```
*   **True Negatives (TN): 965** - The model correctly predicted 965 'ham' messages.
*   **False Positives (FP): 0** - The model incorrectly predicted 0 'ham' messages as 'spam'. This aligns with the 100% precision for 'spam'.
*   **False Negatives (FN): 30** - The model incorrectly predicted 30 'spam' messages as 'ham'. These are the missed spam messages.
*   **True Positives (TP): 120** - The model correctly predicted 120 'spam' messages.

**Model's Strengths:**

*   **Excellent at identifying 'ham' messages:** The model has 100% recall for 'ham', meaning it will not flag legitimate messages as spam. This is crucial for user satisfaction.
*   **Perfect precision for 'spam':** When the model says something is spam, it is always correct (100% precision). This means users won't get annoyed by important 'ham' messages being incorrectly filtered as spam.

**Model's Weaknesses:**

*   **Moderate recall for 'spam':** The main weakness is that it misses 20% of actual spam messages (30 false negatives). While 80% recall is decent, it means some spam will still get through.

**Implications for Text Classification (Spam Detection):**

The Multinomial Naive Bayes model performs very well overall for this dataset. Its high precision for spam and perfect recall for ham are highly desirable in a spam filter. Users will not have legitimate emails incorrectly classified as spam, which is often a major pain point. However, the trade-off is that some spam messages will still slip through. Depending on the application, an 80% recall for spam might be acceptable, or it might require further tuning (e.g., adjusting the classification threshold, trying different models, or adding more features) to capture more spam, potentially at the cost of slight precision reduction for spam or recall for ham. For a first pass, this is a strong and user-friendly spam filter due to its low false positive rate.


## Final Task

### Subtask:
Summarize the entire process, including data extraction, preprocessing, feature engineering, model training, and evaluation results for the Multinomial Naive Bayes classifier.


## Summary:

### Q&A
The entire process involved extracting numerical features from preprocessed text data, splitting this data into training and testing sets, training a Multinomial Naive Bayes classifier, and thoroughly evaluating its performance for a spam detection task.

### Data Analysis Key Findings
*   **Feature Extraction**: Text data was transformed into a TF-IDF feature matrix (X) with 5,572 documents and 5,000 features, using `TfidfVectorizer` with `max_features=5000`. The target variable (y) consisted of 5,572 corresponding labels ('ham' or 'spam').
*   **Data Split**: The dataset was split into training and testing sets with an 80/20 ratio. The training set (`X_train`, `y_train`) contained 4,457 samples, and the testing set (`X_test`, `y_test`) contained 1,115 samples.
*   **Model Performance (Overall)**: The Multinomial Naive Bayes model achieved an overall accuracy of 0.9731 on the test set.
*   **Model Performance ('Ham' Class)**:
    *   The model exhibited perfect recall (1.00) for 'ham' messages, meaning no legitimate messages were incorrectly classified as spam.
    *   Precision for 'ham' was 0.97, and the F1-score was 0.98.
*   **Model Performance ('Spam' Class)**:
    *   The model achieved perfect precision (1.00) for 'spam' messages, indicating that when it predicted a message was spam, it was always correct (no false positives for 'spam').
    *   Recall for 'spam' was 0.80, indicating that 80% of actual spam messages were correctly identified, but 20% were missed.
    *   The F1-score for 'spam' was 0.89.
*   **Confusion Matrix Analysis**:
    *   965 'ham' messages were correctly identified (True Negatives).
    *   0 'ham' messages were incorrectly classified as 'spam' (False Positives).
    *   120 'spam' messages were correctly identified (True Positives).
    *   30 'spam' messages were incorrectly classified as 'ham' (False Negatives), contributing to the 20% missed spam.

### Insights or Next Steps
*   The model is highly effective as a user-friendly spam filter due to its zero false positives for spam (meaning no legitimate messages are incorrectly blocked). This prioritizes user satisfaction by ensuring important communications are never missed.
*   To improve spam detection coverage, future steps could involve exploring techniques to increase the recall for 'spam', such as experimenting with different classifiers (e.g., SVMs, Logistic Regression), tuning hyperparameters, or incorporating more advanced feature engineering, potentially at a slight trade-off for other metrics.
